# Phase 3 — Feature Engineering & Data Management

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
df = pd.read_csv("../data/processed/train_cleaned.csv")
print("Loaded: " + str(df.shape))

Loaded: (1460, 213)


## 1. Feature Engineering

In [2]:
df['TotalSF'] = df['GrLivArea'] + df['TotalBsmtSF']
print("TotalSF created")

TotalSF created


In [3]:
df['HouseAge'] = 2010 - df['YearBuilt']
print("HouseAge created")

HouseAge created


In [4]:
df['TotalBath'] = df['FullBath'] + df['HalfBath'] * 0.5 + df['BsmtFullBath'] + df['BsmtHalfBath'] * 0.5
print("TotalBath created")

TotalBath created


In [5]:
porch_cols = ['WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', 'ScreenPorch']
df['TotalPorchSF'] = df[porch_cols].sum(axis=1)
print("TotalPorchSF created")

TotalPorchSF created


In [6]:
df['YearsSinceRemodel'] = 2010 - df['YearRemodAdd']
print("YearsSinceRemodel created")

YearsSinceRemodel created


## 2. Feature Selection

In [7]:
y = df["SalePrice"]
X = df.drop(columns=["SalePrice"], errors="ignore")
corr = X.corrwith(y).abs().sort_values(ascending=False)
print("Top 15 features by correlation:")
print(corr.head(15).to_string())
keep_cols = corr[corr > 0.1].index.tolist()
print("Kept: " + str(len(keep_cols)) + " features (correlation > 0.1)")

Top 15 features by correlation:
OverallQual     0.790982
TotalSF         0.778959
GrLivArea       0.708624
ExterQual       0.682639
KitchenQual     0.659600
BsmtQual        0.650138
GarageCars      0.640409
TotalBath       0.631731
GarageArea      0.623431
TotalBsmtSF     0.613581
1stFlrSF        0.605852
FullBath        0.560664
TotRmsAbvGrd    0.533723
HouseAge        0.522897
YearBuilt       0.522897
Kept: 98 features (correlation > 0.1)


## 3. Store in SQLite

In [8]:
os.makedirs("../data/processed", exist_ok=True)
import os; os.remove("../data/processed/housing.db") if os.path.exists("../data/processed/housing.db") else None
db_path = "../data/processed/housing.db"
conn = sqlite3.connect(db_path)
X_selected = df[keep_cols + ["SalePrice"]]
X_selected.to_sql("house_prices", conn, if_exists="replace", index=False)
print("Stored " + str(X_selected.shape) + " in SQLite")
print("DB columns:", list(X_selected.columns)[:10], "...")

Stored (1460, 99) in SQLite
DB columns: ['OverallQual', 'TotalSF', 'GrLivArea', 'ExterQual', 'KitchenQual', 'BsmtQual', 'GarageCars', 'TotalBath', 'GarageArea', 'TotalBsmtSF'] ...


## 4. SQL Queries

### Q1: Average price by overall quality rating

In [9]:
q1 = """SELECT OverallQual, ROUND(AVG(SalePrice),0) AS avg_price, COUNT(*) AS cnt
FROM house_prices GROUP BY OverallQual ORDER BY OverallQual DESC"""
print("Q1: Avg Price by Quality")
print(pd.read_sql(q1, conn).to_string(index=False))

Q1: Avg Price by Quality
 OverallQual  avg_price  cnt
          10   438588.0   18
           9   367513.0   43
           8   274736.0  168
           7   207716.0  319
           6   161603.0  374
           5   133523.0  397
           4   108421.0  116
           3    87474.0   20
           2    51770.0    3
           1    50150.0    2


### Q2: Average price by garage car capacity

In [10]:
q2 = """SELECT GarageCars, ROUND(AVG(SalePrice),0) AS avg_price, COUNT(*) AS cnt
FROM house_prices WHERE GarageCars > 0 GROUP BY GarageCars ORDER BY GarageCars DESC"""
print("Q2: Avg Price by Garage Car Capacity")
print(pd.read_sql(q2, conn).to_string(index=False))

Q2: Avg Price by Garage Car Capacity
 GarageCars  avg_price  cnt
          4   192656.0    5
          3   309636.0  181
          2   183852.0  824
          1   128117.0  369


### Q3: Affordable large homes (under $200K, above-average living area)

In [11]:
q3 = """SELECT COUNT(*) AS affordable_large_homes FROM house_prices
WHERE SalePrice < 200000 AND GrLivArea > (SELECT AVG(GrLivArea) FROM house_prices)"""
print("Q3: Affordable Large Homes")
print(pd.read_sql(q3, conn).to_string(index=False))

Q3: Affordable Large Homes
 affordable_large_homes
                    267


### Q4 (Custom): Price by construction era

In [12]:
q4 = """SELECT
  CASE WHEN YearBuilt >= 2000 THEN "2000+" WHEN YearBuilt >= 1970 THEN "1970-1999" ELSE "Pre-1970" END AS era,
  ROUND(AVG(SalePrice),0) AS avg_price, COUNT(*) AS cnt
FROM house_prices GROUP BY era ORDER BY avg_price DESC"""
print("Q4 (Custom): Price by Construction Era")
print(pd.read_sql(q4, conn).to_string(index=False))

Q4 (Custom): Price by Construction Era
      era  avg_price  cnt
    2000+   242439.0  388
1970-1999   188244.0  412
 Pre-1970   140185.0  660


In [13]:
conn.close()
print("Done.")

Done.
